# Benchmark any LLM on real phones — and find the right quant — with TinyEdge

[TinyEdge](https://tinyedge.ai) runs real GGUF models on real edge devices. Two ideas:

1. **Benchmark any model by name.** Point at *any* GGUF on HuggingFace — Llama, Qwen, Gemma, Phi, any size — and the **device downloads it directly**. No local download, no upload, no size cap on your side. You get decode tok/s, time-to-first-token, RAM and on-device perplexity per device.
2. **`optimize=True`** → TinyEdge builds the quantization ladder from your f16, benchmarks every variant on every device as one sweep, and tells you *which quant to ship, per device* — measured, not guessed.

**Before you run:** a [tinyedge.ai](https://tinyedge.ai) account ($25 demo credit; this notebook uses ~$1) · paste your API key in the first code cell · at least one device paired via the TinyEdge Runner app with *"Available for benchmarks"* on · Kaggle **Internet enabled** (right sidebar; needs a phone-verified Kaggle account).

In [ ]:
%pip -q install "tinyedge[hf]>=0.3.0"

In [ ]:
import tinyedge

# Paste your key from tinyedge.ai → New benchmark → "Your API key":
client = tinyedge.TinyEdge(api_key="tinyedge_sk_REPLACE_ME")

DEVICES = client.devices(online=True)     # devices that can claim a job right now
print("benchmarking on:", DEVICES)

## Benchmark any model — just name it

Pass `hf:<owner>/<repo>/<file.gguf>` (or a full huggingface.co URL). The device pulls the
model straight from HuggingFace and runs it — nothing is uploaded from here, so size isn't a
limit. Swap the line below for **any** GGUF: Qwen2.5, Gemma-2, Phi-3, your own repo…

In [ ]:
# A real 1B model (~770 MB) — the device downloads it directly from HuggingFace.
MODEL = "hf:bartowski/Llama-3.2-1B-Instruct-GGUF/Llama-3.2-1B-Instruct-Q4_K_M.gguf"

# one result per device — decode tok/s, time-to-first-token, RAM
client.benchmark(MODEL, devices=DEVICES)

## Find the quant to ship — `optimize=True`

Optimization rewrites the model locally, so this step works from a **local f16 file** (any
GGUF path). It builds the Q8→Q3 ladder, screens out variants that hurt quality, and
benchmarks everything as one sweep — with a WikiText corpus so each variant also reports
on-device perplexity. Here we use a small model so it finishes fast.

In [ ]:
from huggingface_hub import hf_hub_download
F16 = hf_hub_download("bartowski/SmolLM2-135M-Instruct-GGUF", "SmolLM2-135M-Instruct-f16.gguf")

# A WikiText sample for the on-device quality (perplexity) measurement.
import io, zipfile, pathlib, requests
pathlib.Path("corpus").mkdir(exist_ok=True)
wiki = zipfile.ZipFile(io.BytesIO(requests.get(
    "https://huggingface.co/datasets/ggml-org/ci/resolve/main/wikitext-2-raw-v1.zip", timeout=120).content))
pathlib.Path("corpus/wiki.txt").write_bytes(wiki.read("wikitext-2-raw/wiki.test.raw")[:200_000])

report = client.benchmark(F16, devices=DEVICES, dataset="corpus", optimize=True)
print(report.summary())
print("full report with charts + AI analysis:", report.sweep_url)